# ***Building A Chatbot***

In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [ ]:
# Import necessary libraries
import os
from dotenv import load_dotenv

In [2]:
load_dotenv() ## aloading all the environment variable

True

In [ ]:
# Retrieve the Groq API key from environment variables
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
# Import the ChatGroq class from the langchain_groq module
from langchain_groq import ChatGroq

In [ ]:
# Initialize the ChatGroq model with the specified model and API key
model = ChatGroq(model="Gemma2-9b-It", groq_api_key=groq_api_key)

In [ ]:
# Import HumanMessage from langchain_core.messages
from langchain_core.messages import HumanMessage

In [ ]:
# Invoke the model with a HumanMessage to start a conversation
model.invoke([HumanMessage(content="Hi, My name is Emon and I am a Chief AI Engineer")])

AIMessage(content="Hi Emon, it's nice to meet you! That's a fascinating title. \n\nWhat kind of AI projects are you working on? I'm always interested in learning about the cutting edge of AI development.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 23, 'total_tokens': 72, 'completion_time': 0.089090909, 'prompt_time': 0.002204595, 'queue_time': 0.015708574, 'total_time': 0.091295504}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-f682547c-40fe-4b84-9477-5f3e4b110ba0-0', usage_metadata={'input_tokens': 23, 'output_tokens': 49, 'total_tokens': 72})

In [54]:
# Import AIMessage from langchain_core.messages
from langchain_core.messages import AIMessage

In [ ]:
# Invoke the model with a sequence of HumanMessage and AIMessage to simulate a conversation
model.invoke(
    [
        HumanMessage(content="Hi, My name is Emon and I am a Data Scientist"),
        AIMessage(content="Hello Emon! It's nice to meet you. \n\nAs a Data Scientist, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey, What's my name and what do I do?")
    ]
)

AIMessage(content="You are Emon and you are a Data Scientist. 😊 \n\nIs there anything else you'd like to tell me about yourself or your work? I'm happy to chat!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 97, 'total_tokens': 138, 'completion_time': 0.074545455, 'prompt_time': 0.00530831, 'queue_time': 0.016275278, 'total_time': 0.079853765}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-7e572209-113c-47cb-b92e-4fe204a1e7cd-0', usage_metadata={'input_tokens': 97, 'output_tokens': 41, 'total_tokens': 138})

In [55]:
# Import necessary classes for managing chat history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
# Initialize a dictionary to store chat histories
store = {}

# Define a function to get or create a chat history for a given session ID
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Wrap the model with message history functionality
with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [ ]:
# Define a configuration dictionary with a session ID
config = {"configurable": {"session_id": "chat1"}}

In [ ]:
# Invoke the chatbot with a HumanMessage and store the response
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Emon and I am a Data Scientist")],
    config=config
)

In [ ]:
# Display the content of the response
response.content

"Hi Emon, it's nice to meet you!\n\nThat's awesome that you're a Data Scientist! It's a really interesting field. \n\nWhat kind of work do you do as a Data Scientist? Are you working on any cool projects right now?\n"

In [ ]:
# Ask the chatbot to recall the user's name
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Emon. 😊  \n\nHow can I help you today? \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 97, 'total_tokens': 117, 'completion_time': 0.036363636, 'prompt_time': 0.00532413, 'queue_time': 0.017688719, 'total_time': 0.041687766}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-de207b5b-1c1d-4677-818d-2e8074ec3257-0', usage_metadata={'input_tokens': 97, 'output_tokens': 20, 'total_tokens': 117})

In [ ]:
# Change the session ID to create a new chat session
config1 = {"configurable": {"session_id": "chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my name")],
    config=config1
)

In [ ]:
# Display the content of the response
response.content

'As an AI, I have no memory of past conversations and do not know your name. Can you tell me? 😊\n'

In [ ]:
# Update the user's name in the new session
response = with_message_history.invoke(
    [HumanMessage(content="Hey, My name is John")],
    config=config1
)

In [ ]:
# Display the content of the response
response.content

"Hi John! It's nice to meet you. What can I do for you today? 😄  \n\n"

In [ ]:
# Ask the chatbot to recall the user's name in the new session
response = with_message_history.invoke(
    [HumanMessage(content="What's my name")],
    config=config1
)

In [ ]:
# Display the content of the response
response.content

'Your name is John, remember?  \n\nIs there anything else I can help you with? 😊  \n'

In [ ]:
# Import necessary classes for creating a chat prompt template
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
# Define a chat prompt template with a system message and a placeholder for chat messages
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

In [ ]:
# Create a chain by combining the prompt and the model
chain = prompt | model

In [ ]:
# Invoke the chain with a HumanMessage
chain.invoke({"messages": [HumanMessage(content="Hi, My name is Emon")]})

AIMessage(content="Hello Emon, it's nice to meet you! \n\nI'm happy to help with any questions you have.  What can I do for you today? 😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 32, 'total_tokens': 73, 'completion_time': 0.074545455, 'prompt_time': 0.002342156, 'queue_time': 0.016496592999999997, 'total_time': 0.076887611}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-68d4c3c8-54c5-4bed-a493-40a614328432-0', usage_metadata={'input_tokens': 32, 'output_tokens': 41, 'total_tokens': 73})

In [ ]:
# Wrap the chain with message history functionality
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [ ]:
# Define a configuration dictionary with a new session ID
config = {"configurable": {"session_id": "chat3"}}

In [ ]:
# Invoke the chatbot with a HumanMessage and store the response
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Emon")],
    config=config
)

In [ ]:
# Display the response
response

AIMessage(content="Hello Emon! \n\nIt's nice to meet you. How can I assist you today? 😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 128, 'total_tokens': 155, 'completion_time': 0.049090909, 'prompt_time': 0.005546219, 'queue_time': 0.021014138, 'total_time': 0.054637128}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-a96b33c8-234e-44f3-b63a-92b3e977f88d-0', usage_metadata={'input_tokens': 128, 'output_tokens': 27, 'total_tokens': 155})

In [ ]:
# Ask the chatbot to recall the user's name
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

In [ ]:
# Display the content of the response
response.content

'Your name is Emon.  \n\nIs there anything else I can help you with?\n'

In [ ]:
# Add more complexity to the prompt by including a language parameter
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [ ]:
# Create a new chain by combining the updated prompt and the model
chain = prompt | model

In [ ]:
# Invoke the chain with a HumanMessage and specify the language
response = chain.invoke({"messages": [HumanMessage(content="Hi, My name is Emon")], "language": "Hindi"})
response.content

'नमस्ते कृष्ण! मुझे खुशी है कि आप मुझसे जुड़े हैं। मैं आपकी मदद करने के लिए तैयार हूँ। \n\nआप क्या जानना चाहते हैं या क्या करवाना चाहते हैं? 😊 \n'

In [ ]:
# Wrap the new chain with message history functionality
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [ ]:
# Define a configuration dictionary with a new session ID
config = {"configurable": {"session_id": "chat4"}}

In [ ]:
# Invoke the chatbot with a HumanMessage and specify the language
response = with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi, I am Krish")], "language": "Hindi"},
    config=config
)

In [ ]:
# Display the content of the response
response.content

'नमस्ते कृष्ण,  \n\nआप कैसे हैं? मैं आपकी मदद करने के लिए तैयार हूँ। \n\nक्या आप कोई सवाल पूछना चाहते हैं? 😊\n\n'

In [ ]:
# Ask the chatbot to recall the user's name in the specified language
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="What's my name?")], "language": "Hindi"},
    config=config,
)

In [42]:
response.content

'आपका नाम कृष्ण है। 😊 \n\n\n'

In [ ]:
# Import necessary classes for trimming messages
from langchain_core.messages import SystemMessage, trim_messages

In [ ]:
# Define a message trimmer to limit the number of tokens in the conversation
trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [ ]:
# Define a list of messages to simulate a conversation
messages = [
    SystemMessage(content="You're a good assistant"),
    HumanMessage(content="Hi! I'm Bob"),
    AIMessage(content="Hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="Nice"),
    HumanMessage(content="What's 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="Thanks"),
    AIMessage(content="No problem!"),
    HumanMessage(content="Having fun?"),
    AIMessage(content="Yes!"),
]

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\emon1\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In or

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [ ]:
# Trim the messages to fit within the token limit
trimmer.invoke(messages)

In [ ]:
# Import necessary classes for creating a runnable chain
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# Create a chain that trims messages, applies the prompt, and invokes the model
chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

In [ ]:
# Invoke the chain with a HumanMessage and specify the language
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What ice cream do I like?")],
        "language": "English"
    }
)

In [46]:
response.content

"As an AI, I don't have access to your personal information, including your ice cream preferences!  \n\nWhat's your favorite flavor? 😊\n"

In [ ]:
# Invoke the chain with a HumanMessage asking about a previous math problem
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What math problem did I ask?")],
        "language": "English",
    }
)

In [48]:
response.content

'You asked "what is 2 + 2".  😊 \n\n\n'

In [ ]:
# Wrap the chain with message history functionality
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [ ]:
# Define a configuration dictionary with a new session ID
config = {"configurable": {"session_id": "chat5"}}

In [ ]:
# Invoke the chatbot with a HumanMessage asking about the user's name
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="What's my name?")],
        "language": "English",
    },
    config=config,
)

In [51]:
response.content

"As a large language model, I don't have access to past conversations or any personal information about you, including your name.  \n\nIf you'd like to tell me your name, I'd be happy to use it! 😊  \n\n"

In [ ]:
# Invoke the chatbot with a HumanMessage asking about a previous math problem
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="What math problem did I ask?")],
        "language": "English",
    },
    config=config,
)

In [53]:
response.content

"As a large language model, I have no memory of past conversations. If you'd like to ask me a math problem, I'm happy to help! 😊  What is it? \n\n"